# Regulatory Relevance Idenitification for Business Processes
--> retrieving potentially relevant text passages based on input query (process text)

1. semantic search with `SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')`
2. re-ranking with CrossEncoder (`cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')`)

adjusted from:
https://www.sbert.net/examples/applications/retrieve_rerank/README.html

Scores to set expectations - COLIEE results 2023: https://sites.ualberta.ca/~rabelo/COLIEE2023/task1_results.html


In [1]:
!pip install -U sentence-transformers rank_bm25

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 23.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 55.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 32.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 66.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 15.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.0/295.0 kB 34.4 MB/s eta 0:00:00
  Created wheel for sentence-transformers: filename=sentence_transformers-2.2.2-py3-none-any.whl size=125923 sha256=df4c7454c04627986f8436ef6334ee076d9e9e6fba8158c076c641f0fb5ce09b
  Stored in directory: /root/.cache/pip/wheels/62/f2/10/1e606fd5f02395388f74e7462910fe851042f97238cbbd902f
Successfully built sentence-transformers


In [2]:
import json
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import gzip
import os
import torch

if not torch.cuda.is_available():
    print("Warning: No GPU found. Please add GPU to your notebook")

bi_encoder = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')
#reduce long passages to 256 tokens
bi_encoder.max_seq_length = 256
#number of passages retrieved by bi-encoder
top_k = 200

#cross-encoder for re-ranking the results
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')



In [3]:
# Load requirements text data (corpus) for case study 1
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

Saving Input_corpus_uc2.xlsx to Input_corpus_uc2.xlsx
User uploaded file "Input_corpus_uc2.xlsx" with length 45445 bytes


In [4]:
import pandas as pd
df_uc1 = pd.read_excel('Input_corpus_uc2.xlsx')
df_uc1.head()

,requirement_text
0,A reporting entity must take the action set ou...
1,The identity card must\n\n ...
2,The AUSTRAC CEO must issue an identity card to...
3,An authorised officer is not entitled to exe...
4,The pecuniary penalty payable by a body corpor...


In [5]:
# transform to list of strings for case study 1
req_paras_uc1 = df_uc1['requirement_text'].tolist()
print("Passages:", len(req_paras_uc1))
print(req_paras_uc1[0])

Passages: 311
A reporting entity must take the action set out in paragraph 6.1.3 if:

(1)               the reporting entity suspects on reasonable grounds that the customer is not the person that customer claims to be; or

(2)               the reporting entity has doubts about the veracity or adequacy of documents or information previously obtained for the purpose of identifying or verifying:

(a)        the customer; and

(b)       the beneficial owner of the customer (if any); and

(c)        a person purporting to act on behalf of the customer (if any).


In [6]:
# encode all passages into vector space
corpus_embeddings = bi_encoder.encode(req_paras_uc1, convert_to_tensor=True, show_progress_bar=True)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

In [7]:
# preparation of corpus for bm25 algo

from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
import string
from tqdm.autonotebook import tqdm
import numpy as np


# pre-processing corpus: lower case + remove stop-words
def bm25_tokenizer(text):
    tokenized_doc = []
    for token in text.lower().split():
        token = token.strip(string.punctuation)

        if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenized_doc.append(token)
    return tokenized_doc


tokenized_corpus = []
for passage in tqdm(req_paras_uc1):
    tokenized_corpus.append(bm25_tokenizer(passage))

bm25 = BM25Okapi(tokenized_corpus)


  0%|          | 0/311 [00:00<?, ?it/s]

In [8]:
# main relevancy ranking function (based on process info (query)) for bm25

def search_bm25(n, query):

    ##### BM25 (lexical search) + Cross-Encoder #####
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -n)[-n:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)

    # Re-Ranking all retrieved passages with the cross_encoder #
    bm25_cross_inp = [[query, req_paras_uc1[hit['corpus_id']]] for hit in bm25_hits]
    bm25_cross_scores = cross_encoder.predict(bm25_cross_inp)
    # Sort results by the cross-encoder scores
    for idx in range(len(bm25_cross_scores)):
        bm25_hits[idx]['cross-score'] = bm25_cross_scores[idx]

    # Output of top hits from re-ranker
    bm25_hits = sorted(bm25_hits, key=lambda x: x['cross-score'], reverse=True)
    data_list_bm25 = []
    for hit in bm25_hits[0:n]:
        data_list_bm25_new = [query, req_paras_uc1[hit['corpus_id']].replace("\n", " "), hit['cross-score']]
        data_list_bm25.append(data_list_bm25_new)

    return data_list_bm25

In [9]:
# main relevancy ranking function (based on process info (query)) for bi-encoder

def search_bi(n, query):

    ##### Bi-Encoder (semantical search) + Cross-Encoder #####
    # Encoding the query using the bi-encoder and find potentially relevant passages
    question_embedding = bi_encoder.encode(query, convert_to_tensor=True)
    question_embedding = question_embedding.cuda()
    hits = util.semantic_search(question_embedding, corpus_embeddings, top_k=top_k)
    hits = hits[0]  # Get the hits for the first query

    # Re-Ranking all retrieved passages with the cross_encoder #
    cross_inp = [[query, req_paras_uc1[hit['corpus_id']]] for hit in hits]
    cross_scores = cross_encoder.predict(cross_inp)
    # Sort results by the cross-encoder scores
    for idx in range(len(cross_scores)):
        hits[idx]['cross-score'] = cross_scores[idx]

    # Output of top hits from re-ranker
    hits = sorted(hits, key=lambda x: x['cross-score'], reverse=True)
    data_list_bi = []
    for hit in hits[0:n]:
        data_list_new = [query, req_paras_uc1[hit['corpus_id']].replace("\n", " "), hit['cross-score']]
        data_list_bi.append(data_list_new)

    return data_list_bi

# PROCESS LEVEL [1]

In [10]:
# for process level, 49 passages are true relevant, thus the top 100 relevant ranked passages are retrieved

data_list_bm25 = search_bm25(100, query = "Know Your Customer (KYC) regulations for individuals add identity checks to banks’ customer onboarding processes. The KYC process is not fully automated, so it adds to bank employees’ casework, making it one more thing to manage. In the process of creating a new bank account, personal information and documents are gathered from the individual customer, and the name and further personal information are validated through various checks and confirmations. The customer's name is also checked against a Politically Exposed Persons (PEP) database to ensure they are not high-risk individuals, and any suspicious activity is flagged for further investigation. The new customer is manually reviewed following a set of guidelines for acceptability and risk level before being onboarded, and assistance is provided in navigating their account and accessing relevant products and services.")
data_list_bi = search_bi(100, query = "Know Your Customer (KYC) regulations for individuals add identity checks to banks’ customer onboarding processes. The KYC process is not fully automated, so it adds to bank employees’ casework, making it one more thing to manage. In the process of creating a new bank account, personal information and documents are gathered from the individual customer, and the name and further personal information are validated through various checks and confirmations. The customer's name is also checked against a Politically Exposed Persons (PEP) database to ensure they are not high-risk individuals, and any suspicious activity is flagged for further investigation. The new customer is manually reviewed following a set of guidelines for acceptability and risk level before being onboarded, and assistance is provided in navigating their account and accessing relevant products and services.")
print (data_list_bm25)

# create data frames and save to excel
df_1 = pd.DataFrame(data_list_bm25, columns=['query','rel_text','score'])
df_2 = pd.DataFrame(data_list_bi, columns=['query','rel_text','score'])

with pd.ExcelWriter("new_uc2_process_level_algo_output.xlsx") as writer:
    df_1.to_excel(writer, sheet_name="BM25_CE", index=False)
    df_2.to_excel(writer, sheet_name="Bi-Encoder_CE", index=False)

[["Know Your Customer (KYC) regulations for individuals add identity checks to banks’ customer onboarding processes. The KYC process is not fully automated, so it adds to bank employees’ casework, making it one more thing to manage. In the process of creating a new bank account, personal information and documents are gathered from the individual customer, and the name and further personal information are validated through various checks and confirmations. The customer's name is also checked against a Politically Exposed Persons (PEP) database to ensure they are not high-risk individuals, and any suspicious activity is flagged for further investigation. The new customer is manually reviewed following a set of guidelines for acceptability and risk level before being onboarded, and assistance is provided in navigating their account and accessing relevant products and services.", 'An exemption does not apply if the reporting entity determines that it must obtain and verify any KYC informat

# SUBPROCESS LEVEL [2]

In [11]:
# Load process text (query) for case study 1, subprocess level
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

Saving Input_queries_medium_uc2.xlsx to Input_queries_medium_uc2.xlsx
User uploaded file "Input_queries_medium_uc2.xlsx" with length 9731 bytes


In [12]:
df_qm_uc1 = pd.read_excel('Input_queries_medium_uc2.xlsx')
df_qm_uc1.head()

,process_text
0,"To create a new account, the required personal..."
1,A customers identity is validated by verifying...
2,A customer's identity is checked against a Pol...
3,"To verify an account, the accuracy of the cust..."
4,"To manually review a new customer, a thorough ..."


In [14]:
# for the medium process level (7 subprocesses), there are between 3-18 true relevant passages for each subprocess with an average of 11 passages over the 7, thus the top 30 relevant ranked passages are retrieved

# transform query input to list of strings
queries_m_uc1 = df_qm_uc1['process_text'].tolist()
print("Passages:", len(queries_m_uc1))

# iterate through all queries for use case 1
all_query_data_list_bm25 = []
all_query_data_list_bi = []
for query in queries_m_uc1:
  data_list_bm25_new = search_bm25(30, query)
  all_query_data_list_bm25.extend(data_list_bm25_new)
  data_list_bi_new = search_bi(30, query)
  all_query_data_list_bi.extend(data_list_bi_new)

# create data frames and save to excel
df_1 = pd.DataFrame(all_query_data_list_bm25, columns=['query','rel_text','score'])
df_2 = pd.DataFrame(all_query_data_list_bi, columns=['query','rel_text','score'])

with pd.ExcelWriter("new_uc2_subprocess_level_algo_output.xlsx") as writer:
    df_1.to_excel(writer, sheet_name="BM25_CE", index=False)
    df_2.to_excel(writer, sheet_name="Bi-Encoder_CE", index=False)

Passages: 7


# TASK/EVENT LEVEL [3]

In [15]:
# Load process text (query) for case study 1, subprocess level
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

Saving Input_queries_low_uc2.xlsx to Input_queries_low_uc2.xlsx
User uploaded file "Input_queries_low_uc2.xlsx" with length 9714 bytes


In [16]:
df_ql_uc1 = pd.read_excel('Input_queries_low_uc2.xlsx')
df_ql_uc1.head()

,process_text
0,Collect all the necessary personal information...
1,Enter the customer's information into the bank...
2,Create a unique account number for the new cus...
3,Verify the customer's full name using a govern...
4,Verify the customer's required personal inform...


In [17]:
# for the detailed process level (31 tasks/throwing events), there are between 0-9 true relevant passages for each task with an average of 4 passages over the 31, thus the top 15 relevant ranked passages are retrieved

# transform query input to list of strings
queries_l_uc1 = df_ql_uc1['process_text'].tolist()
print("Passages:", len(queries_l_uc1))

# iterate through all queries for use case 1
all_query_data_list_bm25 = []
all_query_data_list_bi = []
for query in queries_l_uc1:
  data_list_bm25_new = search_bm25(15, query)
  all_query_data_list_bm25.extend(data_list_bm25_new)
  data_list_bi_new = search_bi(15, query)
  all_query_data_list_bi.extend(data_list_bi_new)

# create data frames and save to excel
df_1 = pd.DataFrame(all_query_data_list_bm25, columns=['query','rel_text','score'])
df_2 = pd.DataFrame(all_query_data_list_bi, columns=['query','rel_text','score'])

with pd.ExcelWriter("new_uc2_event_level_algo_output.xlsx") as writer:
    df_1.to_excel(writer, sheet_name="BM25_CE", index=False)
    df_2.to_excel(writer, sheet_name="Bi-Encoder_CE", index=False)

Passages: 19
